# Stage1 10:32 OFR Funding 压力：0bp恢复净优势信息检验

旧21项恢复特征为基线，只增加保守滞后且按官方修订旧值复原的 Funding 坐标。预设系数为负，完整动作日历和0bp Tiered净优势标签保持一致。

In [1]:
import hashlib
import json
import os
import shlex
from pathlib import Path

import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from src.analysis.seed_disagreement_information import full_calendar_bootstrap
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import _ssh, archive_nas_recorder
from src.workflow.intraday_recovery_direct_allocation import FEATURE_COLUMNS

os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
root=Path.cwd().resolve();card=root/'research/stage1_ofr_funding_0bp_action_information_20261023'
plan=json.loads((card/'plan.json').read_text());source=plan['sources']
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['execution_authorized_by_active_Goal'] and not (card/'run_state.json').exists()
assert plan['budget']=={'new_external_requests':0,'source_Recorder_object_loads_cumulative_before_fits_max':4,'DatasetH_builds_cumulative_before_fits_max':2,'model_fit_attempts_max':2,'bootstrap_attempts_max':1,'new_writable_Qlib_Recorders_max':1,'native_accounts':0}
action_card=root/source['action_feature_card'];action_summary=json.loads((action_card/'summary.json').read_text());action_binding=json.loads((action_card/'nas_binding.json').read_text())
assert sha(action_card/'summary.json')==source['action_feature_summary_sha256'] and action_summary['recorder_id']==source['action_feature_recorder_id']
assert action_summary['nas_evidence']['sha256']==action_binding['evidence']['sha256']==source['action_feature_NAS_sha256'] and action_summary['nas_readback']['verified']
action_artifact=Path(action_summary['recorder_uri'])/action_summary['experiment_id']/action_summary['recorder_id']/'artifacts'/source['action_feature_artifact'];assert sha(action_artifact)==source['action_feature_artifact_sha256']
c=root/source['label_card'];s=json.loads((c/'summary.json').read_text());m=json.loads((c/'verification_manifest.json').read_text());b=json.loads((c/'nas_binding.json').read_text())
assert sha(c/'summary.json')==source['label_summary_sha256'] and sha(c/'verification_manifest.json')==source['label_manifest_sha256']
assert s['recorder_id']==source['label_recorder_id'] and s['nas_evidence']['sha256']==b['evidence']['sha256']==source['label_NAS_sha256']
restore=json.loads((c/'nas_scoped_restore.json').read_text());assert restore['status']=='verified' and restore['archive_sha256']==source['label_NAS_sha256']
matches=[c/e['path'] for e in m['entries'] if e['path'].endswith('/analysis/labels')]
assert len(matches)==1 and sha(matches[0])==source['label_artifact_sha256']
label_plan=json.loads((c/'plan.json').read_text());module='src/workflow/intraday_recovery_direct_allocation.py'
assert sha(c/'plan.json')==s['plan_sha256'] and sha(root/module)==label_plan['code_sha256'][module], 'frozen 21-recovery-feature module SHA mismatch'
qlib.init(provider_uri=str(root/'data/qlib/full_path_tiered_allocation_20261013'),region='us')
assert len(D.calendar(start_time='2018-10-15',end_time='2022-12-30',freq='day'))==plan['cohorts']['calendar_rows']
def load(prefix,artifact):
    c=root/source[prefix+'_card']
    with R.uri_context(str(c/'qlib_recorder')):
        return R.get_recorder(experiment_name=c.name,recorder_id=source[prefix+'_recorder_id']).load_object(artifact)
feature=load('action_feature',source['action_feature_artifact'])
labels=load('label','analysis/labels')
old_names=list(FEATURE_COLUMNS);assert len(old_names)==21 and len(set(old_names))==21
assert len(labels)==plan['cohorts']['calendar_rows'] and labels.trade_date.is_unique
coord=feature[['trade_date','funding_stress_last_conservatively_known','source_known','assumed_available_at']]
coord['source_status']=np.where(coord.source_known,'known','unknown')
joined=labels.merge(coord,on='trade_date',validate='one_to_one',sort=False)
assert len(joined)==len(labels) and joined.trade_date.equals(labels.trade_date)
assert joined.source_status.eq('known').mean()>=0.99 and int((joined.eligible&joined.source_status.ne('known')).sum())<=2
decision_local=pd.to_datetime(joined.decision_at).dt.tz_convert('America/New_York').dt.tz_localize(None)
assert (pd.to_datetime(joined.assumed_available_at)<=decision_local).all()
joined['feature_available_before_decision']=pd.to_datetime(joined.assumed_available_at)<=decision_local
joined['label_mature']=joined.net_advantage.notna()&pd.to_datetime(joined.label_available_at).notna()
assert (pd.to_datetime(joined.loc[joined.label_mature,'label_available_at'])>pd.to_datetime(joined.loc[joined.label_mature,'decision_at'])).all()
cuttrain=pd.Timestamp(plan['cohorts']['train_label_mature_before']);cuteval=pd.Timestamp(plan['cohorts']['evaluation_label_mature_before'])
train_calendar=joined.trade_date.between(*plan['cohorts']['train_trade_dates'])
eval_calendar=joined.trade_date.between(*plan['cohorts']['evaluation_trade_dates'])
train_mask=train_calendar&joined.eligible&joined.label_mature&pd.to_datetime(joined.label_available_at).lt(cuttrain)
eval_mask=eval_calendar&joined.eligible&joined.label_mature&pd.to_datetime(joined.label_available_at).lt(cuteval)
assert train_calendar.sum()+eval_calendar.sum()==len(joined)
assert int(train_mask.sum())>=plan['cohorts']['minimum_mature_train_eligible'] and int(eval_mask.sum())>=plan['cohorts']['minimum_mature_eval_eligible']
by_eval_year={int(y):int((eval_mask&joined.trade_date.dt.year.eq(y)).sum()) for y in plan['cohorts']['evaluation_complete_years']}
assert all(x>=plan['cohorts']['minimum_each_eval_year'] for x in by_eval_year.values())
assert not np.isinf(joined.loc[train_mask|eval_mask,'funding_stress_last_conservatively_known'].to_numpy(float)).any()
static_features=joined[old_names+['funding_stress_last_conservatively_known']].copy()
meta=pd.DataFrame({'eligible':joined.eligible.to_numpy(bool),'label_available_at':joined.label_available_at,'train_scored':train_mask.to_numpy(bool),'eval_scored':eval_mask.to_numpy(bool),'source_known':joined.source_status.eq('known').to_numpy(bool)})
static=pd.concat({'feature':static_features,'label':joined[['net_advantage']],'meta':meta},axis=1)
static.index=pd.MultiIndex.from_arrays([pd.DatetimeIndex(joined.trade_date),['SOXL_SOXX_CASH']*len(joined)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(plan['cohorts']['train_trade_dates']),'eval':tuple(plan['cohorts']['evaluation_trade_dates'])})
tr=dataset.prepare('train',col_set=['feature','label','meta']);he=dataset.prepare('eval',col_set=['feature','label','meta'])
train=tr.loc[tr[('meta','train_scored')].astype(bool)];held=he.loc[he[('meta','eval_scored')].astype(bool)]
assert len(train)==int(train_mask.sum()) and len(held)==int(eval_mask.sum())
assert (train[('meta','eligible')].astype(bool).all() and held[('meta','eligible')].astype(bool).all())
ytr=train[('label','net_advantage')].to_numpy(float);yhe=held[('label','net_advantage')].to_numpy(float)
assert np.isfinite(ytr).all() and np.isfinite(yhe).all()
preflight={'status':'fixed_0bp_action_value_DatasetH_preflight_passed','plan_sha256':sha(card/'plan.json'),'calendar_rows':len(joined),'train_calendar_rows':int(train_calendar.sum()),'eval_calendar_rows':int(eval_calendar.sum()),'train_mature_eligible':len(train),'eval_mature_eligible':len(held),'eval_annual_support':by_eval_year,'unknown_source_days':int(joined.source_status.ne('known').sum()),'eligible_unknown_source_days':int((joined.eligible&joined.source_status.ne('known')).sum()),'new_model_fits':0,'native_accounts':0}
preflight['max_abs_training_pairwise_Pearson']=float(train['feature'][old_names].corrwith(train['feature']['funding_stress_last_conservatively_known']).abs().max())
assert preflight['max_abs_training_pairwise_Pearson']<plan['receiver']['training_only_max_abs_pairwise_Pearson_stop_at_or_above']
receipt=card/'preflight_result.json'
if receipt.exists():assert json.loads(receipt.read_text())==json.loads(json.dumps(preflight))
else:atomic_json(preflight,receipt)


2026/10/23 04:57:49 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[87258:MainThread](2026-10-23 04:57:50,964) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[87258:MainThread](2026-10-23 04:57:50,966) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[87258:MainThread](2026-10-23 04:57:50,966) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013')}


/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_87258/1894128648.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  coord['source_status']=np.where(coord.source_known,'known','unknown')
[87258:MainThread](2026-10-23 04:57:51,042) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[87258:MainThread](2026-10-23 04:57:51,042) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[87258:MainThread](2026-10-23 04:57:51,042) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


## 独立设计审阅门

无拟合预检通过后，审阅准确来源、计划和拟合单元；首次写入 Recorder 前执行一次预算门。

In [2]:
review_path=card/'design_review_final.json';assert review_path.exists()
review=json.loads(review_path.read_text());notebook=json.loads((card/(card.name+'.ipynb')).read_text());preflight_hash=hashlib.sha256(''.join(notebook['cells'][1]['source']).encode()).hexdigest();fit_hash=hashlib.sha256(''.join(notebook['cells'][3]['source']).encode()).hexdigest()
assert review['verdict']=='APPROVE_DESIGN' and review['plan_sha256']==sha(card/'plan.json') and review['preflight_sha256']==sha(card/'preflight_result.json') and review['preflight_cell_sha256']==preflight_hash and review['fit_cell_sha256']==fit_hash
assert not (card/'run_state.json').exists() and not (card/'qlib_recorder').exists(), 'fixed fit budget already consumed or writable Recorder exists'
state={'status':'fit_attempting','plan_sha256':sha(card/'plan.json'),'fit_attempts':[],'fits_completed':[],'bootstrap_attempts':0,'native_accounts':0}
atomic_json(state,card/'run_state.json')
inputs={'baseline':(train['feature'][old_names].to_numpy(float),held['feature'][old_names].to_numpy(float)),'candidate':(train['feature'][old_names+['funding_stress_last_conservatively_known']].to_numpy(float),held['feature'][old_names+['funding_stress_last_conservatively_known']].to_numpy(float))}
models={};predictions={}
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
    recorder=R.get_recorder();state.update(recorder_id=recorder.id,experiment_id=str(recorder.experiment_id));atomic_json(state,card/'run_state.json')
    dataset.config(dump_all=True,recursive=True)
    recorder.save_objects(artifact_path='inputs',plan=plan,preflight=preflight,dataset=dataset,source_recorder_ids={k:source[k+'_recorder_id'] for k in ('action_feature','label')})
    for family in ('baseline','candidate'):
        assert len(state['fit_attempts'])<2 and family not in state['fit_attempts']
        state['fit_attempts'].append(family);atomic_json(state,card/'run_state.json')
        model=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0))
        model.fit(inputs[family][0],ytr)
        prediction=model.predict(inputs[family][1])
        assert np.isfinite(prediction).all()
        models[family]=model;predictions[family]=prediction
        recorder.save_objects(artifact_path='models',**{family:model})
        recorder.save_objects(artifact_path='predictions',**{family:pd.DataFrame({'trade_date':pd.DatetimeIndex(held.index.get_level_values('datetime')),'prediction':prediction,'target':yhe})})
        state['fits_completed'].append(family);atomic_json(state,card/'run_state.json')
        recorder.log_metrics(completed_fits=len(state['fits_completed']))
    dates=pd.DatetimeIndex(held.index.get_level_values('datetime'))
    scored=pd.DataFrame({'trade_date':dates,'target':yhe,'baseline_prediction':predictions['baseline'],'candidate_prediction':predictions['candidate']})
    scored['paired_mse_improvement']=(scored.target-scored.baseline_prediction)**2-(scored.target-scored.candidate_prediction)**2
    full=pd.DataFrame({'trade_date':joined.loc[eval_calendar,'trade_date'].to_numpy(),'paired_mse_improvement':np.nan})
    full=full.merge(scored,on='trade_date',how='left',suffixes=('_calendar',''),validate='one_to_one')
    full['paired_mse_improvement']=full['paired_mse_improvement'].fillna(full.pop('paired_mse_improvement_calendar'))
    assert len(full)==int(eval_calendar.sum()) and int(full.paired_mse_improvement.notna().sum())==len(held)
    annual=[{'year':int(year),'calendar_rows':len(g),'scored_rows':int(g.paired_mse_improvement.notna().sum()),'mean_paired_mse_improvement':float(g.paired_mse_improvement.mean())} for year,g in full.groupby(full.trade_date.dt.year,sort=True)]
    assert [x['year'] for x in annual]==plan['cohorts']['evaluation_complete_years']
    boot=plan['information_gate']['paired_stationary_bootstrap'];state['bootstrap_attempts']=1;atomic_json(state,card/'run_state.json')
    interval,indices,replicates=full_calendar_bootstrap(full.paired_mse_improvement.to_numpy(float),mean_block=boot['mean_block_sessions'],resamples=boot['draws'],seed=boot['seed'],alpha=boot['one_sided_alpha'])
    assert interval['valid_replicates']>=9990
    coefficient=float(models['candidate'].named_steps['ridge'].coef_[-1])
    metrics={'baseline_eval_mse':float(np.mean((scored.target-scored.baseline_prediction)**2)),'candidate_eval_mse':float(np.mean((scored.target-scored.candidate_prediction)**2)),'paired_mse_improvement':float(scored.paired_mse_improvement.mean()),'annual':annual,'added_standardized_coefficient':coefficient,'bootstrap_lower':interval['lower_bound'],'bootstrap_upper':interval['upper_bound'],'bootstrap':interval}
    gates={'coefficient_negative':coefficient<0,'full_eval_mean_positive':metrics['paired_mse_improvement']>0,'each_eval_year_mean_positive':all(x['mean_paired_mse_improvement']>0 for x in annual),'bootstrap_lower_positive':metrics['bootstrap_lower']>0}
    passed=bool(all(gates.values()))
    recorder.save_objects(artifact_path='analysis',scored=scored,full_eval_calendar=full,metrics=metrics,gates=gates,bootstrap_replicates=replicates,bootstrap_indices=indices)
    recorder.log_metrics(paired_mse_improvement=metrics['paired_mse_improvement'],bootstrap_lower=metrics['bootstrap_lower'],information_gate_pass=int(passed))
    recorder_id,experiment_id=recorder.id,str(recorder.experiment_id)
summary={'experiment':plan['experiment'],'decision':'conditional_development_OFR_Funding_action_information_pass_source_receipt_pending_no_account' if passed else 'not_promoted_fixed_0bp_OFR_Funding_recovery_amount_information_failed','objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'preflight':preflight,'information_gate_pass':passed,'metrics':metrics,'gates':gates,'new_model_fits':2,'new_bootstraps':1,'new_native_accounts':0,'evaluated_held_2023_2026':False,'historical_independence':False,'recorder_id':recorder_id,'experiment_id':experiment_id,'recorder_uri':str(card/'qlib_recorder'),'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','models/*','predictions/*','analysis/*']},'limitations':['2018-2022 OOF Stage1-architecture local action labels are reused development history and not independent forward evidence.','The training reference omits actual broker-fill and corporate-action parity; no 2023+ complete funded account was run.','The OFR Funding category uses a conservative three-Qlib-session lag and documented old revision values, but its exact historical daily first receipt and finality remain unqualified; current-method pre-2017 backfill is excluded.']}
atomic_json(summary,card/'summary.json')
binding=json.loads((root/source['label_card']/'nas_binding.json').read_text());binding.pop('evidence',None);binding.update(source_id=plan['experiment'],source_plan='plan.json',as_of='2022-12-30')
atomic_json(binding,card/'nas_binding.json')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=recorder_id,experiment_id=experiment_id)
remote=_ssh(binding['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0]
assert remote==evidence['sha256']
summary['nas_evidence']=evidence;summary['nas_readback']={'sha256':remote,'bytes':evidence['size_bytes'],'verified':True}
atomic_json(summary,card/'summary.json')
state.update(status='completed',recorder_id=recorder_id,experiment_id=experiment_id)
atomic_json(state,card/'run_state.json')
print(summary['decision'],preflight['train_mature_eligible'],preflight['eval_mature_eligible'],metrics,gates,recorder_id)


[87258:MainThread](2026-10-23 04:57:51,057) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_ofr_funding_0bp_action_information_20261023.


[87258:MainThread](2026-10-23 04:57:51,058) INFO - qlib.workflow - [exp.py:258] - Experiment 760365360549602005 starts running ...


[87258:MainThread](2026-10-23 04:57:51,127) INFO - qlib.workflow - [recorder.py:345] - Recorder 9818982b88d94b7fad413be2912ff6e7 starts running under Experiment 760365360549602005 ...


[87258:MainThread](2026-10-23 04:57:52,991) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


not_promoted_fixed_0bp_OFR_Funding_recovery_amount_information_failed 173 265 {'baseline_eval_mse': 8.382540986858794e-05, 'candidate_eval_mse': 7.757045582696983e-05, 'paired_mse_improvement': 6.254954041618131e-06, 'annual': [{'year': 2021, 'calendar_rows': 252, 'scored_rows': 27, 'mean_paired_mse_improvement': -4.471409561774802e-06}, {'year': 2022, 'calendar_rows': 251, 'scored_rows': 238, 'mean_paired_mse_improvement': 7.471810416792959e-06}], 'added_standardized_coefficient': 0.0006998612524085787, 'bootstrap_lower': -4.495299831827363e-06, 'bootstrap_upper': 1.241127102217228e-05, 'bootstrap': {'paired_mean_loss_improvement': 6.2549540416181315e-06, 'lower_bound': -4.495299831827363e-06, 'upper_bound': 1.241127102217228e-05, 'valid_replicates': 10000, 'invalid_replicates': 0, 'calendar_rows': 503, 'scored_rows': 265}} {'coefficient_negative': False, 'full_eval_mean_positive': True, 'each_eval_year_mean_positive': False, 'bootstrap_lower_positive': False} 9818982b88d94b7fad413be2

# Stage1 10:32 OFR Funding：0bp恢复净优势信息检验

结论：`not_promoted_fixed_0bp_OFR_Funding_recovery_amount_information_failed`。冻结的 2018–2020 训练、2021–2022 评价上，旧恢复 21 项输入追加唯一 OFR Funding 压力坐标后，局部 0bp Tiered beta0.6 恢复/HOLD 金额 MSE 整体改善，但未通过预登记的负系数、两个评价年和校正区块下界门。此版关闭，不反转符号、调整滞后、修订口径、年份或阈值补救。

Qlib DatasetH 保留全部 1,061 个动作日；成熟合格训练 173 条、评价 265 条，2021 年 27 条、2022 年 238 条。两个来源未知日均不在合格标签内；训练期 Funding 与旧输入最大单项绝对相关 0.695。执行两次固定 Ridge、一次 10,000 次全日历区块检验，0 新账户。

评价 MSE：基线 8.38254099e-05，候选 7.75704558e-05；成对改善 +6.25495404e-06。2021/2022 改善 -4.47140956e-06/+7.47181042e-06；新增标准化系数 +0.000699861252，与预设的负方向相反；固定 63 日平均块、单侧 0.025/33 下界 -4.49529983e-06，仍为负。仅整体 MSE 门通过。

来源先剔除现行 OFR FSI 2017 年发布前回填；每个动作日只看第三个前序 Qlib 交易日前的官方 Funding，2022 年使用官方修订工作簿旧值。今日官网与工作簿不认证每个历史日的首次接收和未列明修订；训练参考也不是 2023 年以后的原始 Stage1 实物账户。不能用 2022 年局部改善或反向系数宣称交易收益。

初次无拟合预检因模板把来源工件写成 `analysis/daily` 而在读对象前停止；保留失败记录后只改为已冻结 SHA 的 `analysis/mapped_signals`，无额外拟合。独立审阅按修复后的准确版本批准。

原生 Qlib Recorder `9818982b88d94b7fad413be2912ff6e7` 的模型、DatasetH、指标和门槛已目标读回；NAS SHA-256 `f25b9f71157be07e5e08eec302458819f24d7f4fe2e8138e89cc308a028c3601` 经限定恢复四件核心工件逐一核对。下一步继续找能同时应对原 Stage1 现金机会和 SOXL 相对选择失败的**不同**决策时信息；任何候选须先在完整原 Stage1 派生 0bp 账户过 SOXX 硬门，随后独立做 2/5bp 与前向验证。